In [1]:
#Apply PCA & Otliers removal
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN, SpectralClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.cluster import Birch
from sklearn.cluster import OPTICS
from scipy.stats import zscore
import numpy as np
from sklearn.decomposition import PCA
import time
# Load dataset
df= pd.read_csv("data/data.csv")

In [2]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df['diagnosis'] = le.fit_transform(df['diagnosis'])
df.isnull().sum().sum()
df = df.dropna(axis=1, how='all')
# Drop target and ID columns
X = df.drop(columns=["id"], errors="ignore")
print("Features shape:", X.shape)

Features shape: (569, 31)


In [ ]:
# z_scores = np.abs(zscore(X))

# # Remove rows where any feature has z > threshold (e.g., 3)
# threshold = 3
# # mask = (z_scores < threshold).all(axis=1)
# mask = (z_scores < threshold).mean(axis=1) > 0.95
# X_out = X[mask]

# print("Shape after outlier removal:", X_out.shape)


Shape after outlier removal: (522, 31)


In [7]:
# Compute z-scores
z_scores = np.abs((X - X.mean()) / X.std())

# Define threshold
threshold = 3
# Get row indices where ANY feature is an outlier
outlier_indices = np.where((z_scores > threshold).any(axis=1))[0]

# Remove them
X_out = X.drop(index=X.index[outlier_indices])
print("Original features shape:", X.shape)
print("After Outlier Removal:", X_out.shape)


Original features shape: (569, 31)
After Outlier Removal: (495, 31)


In [8]:
#apply scaling
scaler = StandardScaler()
X_so = scaler.fit_transform(X_out)

#apply PCA
pca = PCA(n_components=0.95, random_state=42)
X_sop = pca.fit_transform(X_so)#Apply PCA
#print("PCA features shape:", X_pca.shape)
print("PCA-reduced features shape:", X_sop.shape)
print("Explained variance ratio sum:", sum(pca.explained_variance_ratio_))

PCA-reduced features shape: (495, 11)
Explained variance ratio sum: 0.9582635566861605


In [9]:
#Define Clustering Parameters
k_values = range(2, 9)  # clusters for KMeans, GMM, Agglomerative, Spectral
n_init = 10              # random initialization
dbscan_eps = [0.5, 1.0, 1.5]  # DBSCAN eps values
min_samples = 5

#Function to Compute Metrics
def compute_metrics(X_data, labels):
    sil = silhouette_score(X_data, labels)
    db = davies_bouldin_score(X_data, labels)
    ch = calinski_harabasz_score(X_data, labels)
    return sil, db, ch

In [10]:
#K-Means on Scaled + PCA + outliersremoval Data
start_time = time.time()
kmean_pca_out = []
for k in k_values:
    km = KMeans(n_clusters=k, n_init=n_init, random_state=42)
    labels = km.fit_predict(X_sop)
    sil, db, ch = compute_metrics(X_sop, labels)
    kmean_pca_out.append({"algorithm": "KMeans", "preprocessing": "PCA+Outliers", "k": k, "silhouette": sil, "davies_bouldin": db, "calinski_harabasz": ch})


end_time = time.time()
runtime = end_time - start_time
# avg_time = np.mean(times)
print("Runtime:", runtime, "seconds")
print(f"K-means runtime: {runtime:.4f} seconds")  

Runtime: 4.250381708145142 seconds
K-means runtime: 4.2504 seconds


In [11]:
#Gaussian Mixture (GMM)on Scaled + PCA + outliersremoval Data
start_time = time.time()
gmm_pca_out = []
for k in k_values:
    gmm = GaussianMixture(n_components=k, n_init=n_init, random_state=42)
    labels = gmm.fit_predict(X_sop)
    sil, db, ch = compute_metrics(X_sop, labels)
    gmm_pca_out.append({"algorithm": "GMM", "preprocessing": "PCA+Outliers", "k": k, "silhouette": sil, "davies_bouldin": db, "calinski_harabasz": ch})

end_time = time.time()
runtime = end_time - start_time
# avg_time = np.mean(times)
print("Runtime:", runtime, "seconds")
print(f"GMM runtime: {runtime:.4f} seconds")    

Runtime: 2.819035053253174 seconds
GMM runtime: 2.8190 seconds


In [12]:
#Agglomerative Clustering on Scaled + PCA + outliersremoval Data
start_time = time.time()
agg_pca_out = []
for k in k_values:
    agg = AgglomerativeClustering(n_clusters=k, linkage="ward")
    labels = agg.fit_predict(X_sop)
    sil, db, ch = compute_metrics(X_sop, labels)
    agg_pca_out.append({"algorithm": "Agglomerative", "preprocessing": "PCA+Outliers", "k": k, "silhouette": sil, "davies_bouldin": db, "calinski_harabasz": ch})
    
end_time = time.time()
runtime = end_time - start_time
# avg_time = np.mean(times)
print("Runtime:", runtime, "seconds")
print(f"Agglomerative runtime: {runtime:.4f} seconds")

Runtime: 0.3338446617126465 seconds
Agglomerative runtime: 0.3338 seconds


In [13]:
#Spectral Clustering on Scaled + PCA + outliersremoval Data
start_time = time.time()
spec_pca_out = []
for k in k_values:
    spec = SpectralClustering(n_clusters=k, affinity="nearest_neighbors")
    labels = spec.fit_predict(X_sop)
    sil, db, ch = compute_metrics(X_sop, labels)
    spec_pca_out.append({"algorithm": "Spectral", "preprocessing": "PCA+Outliers", "k": k, "silhouette": sil, "davies_bouldin": db, "calinski_harabasz": ch})
end_time = time.time()
runtime = end_time - start_time
# avg_time = np.mean(times)
print("Runtime:", runtime, "seconds")
print(f"Spectral runtime: {runtime:.4f} seconds")    

Runtime: 0.5614550113677979 seconds
Spectral runtime: 0.5615 seconds


In [14]:
#DBSCAN on Scaled + PCA + outliersremoval Data
start_time = time.time()
dbscan_pca_out = []
for eps in dbscan_eps:
    dbscan = DBSCAN(eps=eps, min_samples=min_samples)
    labels = dbscan.fit_predict(X_sop)
    
    # Remove noise points (-1)
    mask = labels != -1
    if np.sum(mask) > 1:  # silhouette requires >= 2 points
        sil, db, ch = compute_metrics(X_sop[mask], labels[mask])
        dbscan_pca_out.append({"algorithm": "DBSCAN", "preprocessing": "PCA+Outliers", "eps": eps, "silhouette": sil, "davies_bouldin": db, "calinski_harabasz": ch})
end_time = time.time()
runtime = end_time - start_time
# avg_time = np.mean(times)
print("Runtime:", runtime, "seconds")
print(f"DBSCAN runtime: {runtime:.4f} seconds")

Runtime: 0.021392107009887695 seconds
DBSCAN runtime: 0.0214 seconds


In [16]:
#BIRCH on Scaled + PCA + outliersremoval Data
start_time = time.time()
birch_pca_out = []
threshold_values = [0.2, 0.5, 1.0, 1.5]

for t in threshold_values:
    birch = Birch(n_clusters=None, threshold=t)
    labels = birch.fit_predict(X_sop)

    n_clusters = len(set(labels))
    if 1 < n_clusters < len(X_so) and len(set(labels[mask])) > 1:
        sil, db, ch = compute_metrics(X_so, labels)
        birch_pca_out.append({
            "algorithm": "BIRCH",
            "preprocessing": "PCA+Outliers",
            "threshold": t,
            "n_clusters": len(set(labels)),
            "silhouette": sil,
            "davies_bouldin": db,
            "calinski_harabasz": ch
        })

end_time = time.time()
runtime = end_time - start_time
# avg_time = np.mean(times)
print("Runtime:", runtime, "seconds")
print(f"BIRCH runtime: {runtime:.4f} seconds")

Runtime: 0.38710784912109375 seconds
BIRCH runtime: 0.3871 seconds


In [17]:
#OPTICS on Scaled + PCA + outliersremoval Data
start_time = time.time()
optics_pca_out = []
min_samples_values = [3, 5, 10, 20]

for m in min_samples_values:
    optics = OPTICS(min_samples=m, xi=0.05, n_jobs=-1)
    labels = optics.fit_predict(X_sop)

    # Remove noise points (-1) if needed
    unique_labels = set(labels) - {-1}

    if len(unique_labels) > 1:
        sil, db, ch = compute_metrics(X_sop, labels)
        optics_pca_out.append({
            "algorithm": "OPTICS",
            "preprocessing": "PCA+Outliers",
            "min_samples": m,
            "xi": 0.05,
            "n_clusters": len(unique_labels),
            "silhouette": sil,
            "davies_bouldin": db,
            "calinski_harabasz": ch
        })

end_time = time.time()
runtime = end_time - start_time
# avg_time = np.mean(times)
print("Runtime:", runtime, "seconds")
print(f"OPTICS runtime: {runtime:.4f} seconds")

Runtime: 62.69169998168945 seconds
OPTICS runtime: 62.6917 seconds


In [ ]:
import csv


breast_cancer_results_out_pca = (kmean_pca_out + gmm_pca_out + agg_pca_out + spec_pca_out + dbscan_pca_out+birch_pca_out + optics_pca_out)

keys = ["algorithm", "preprocessing","k", "eps", "min_samples", "threshold","n_clusters","silhouette", "davies_bouldin", "calinski_harabasz"]

with open('updated_data/breast_cancer_data/breast_cancer_outliers_pca.csv', 'w', newline='') as file:
    writer = csv.DictWriter(file, fieldnames=keys, extrasaction="ignore")
    writer.writeheader()
    writer.writerows(breast_cancer_results_out_pca)